# Fase 5 — Integração: financeiro, verificação e avaliação

**Problema desta fase:** fechar o agente único com o financeiro auditado, verificar as respostas antes de mostrar e **avaliar** o agente por qualidade da resposta **e** por trajetória das ferramentas.

**Conceitos da disciplina demonstrados:** verificação de resposta (regras determinísticas + LLM como juiz com uma tentativa de reparo), dataset de avaliação com pergunta, referência e ferramentas esperadas, avaliador de trajetória por igualdade exata, juiz semântico, teste do próprio juiz contra falso positivo ("Posso consultar…" recebe zero).

**Decisões principais**
- Saldo = orçamento **aceito** − pagamentos **confirmados**, calculado em Python. Pagamento apenas informado não altera saldo; estorno é auditado e preserva o lançamento.
- O verificador reduz, mas não elimina, erros: "verificação aprovada" não prova que a fonte estava correta.
- Interface Streamlit (`app.py`) usa os mesmos serviços e o mesmo SQLite do terminal; a escolha de usuário é demonstração, não autenticação.

**Execução real:** a avaliação com o agente real é feita por `python avaliar.py --modo api` (e `--langsmith` para o experimento). Resultado de 4 execuções reais (12/11/2026): respostas 16/16, trajetória exata 12/16, e um falso positivo do juiz contido pela regra de composição. Análise completa em [avaliacao_resultados.md](../docs/avaliacao_resultados.md); este notebook não substitui essa execução. Guia: [fase5_interface_verificacao_avaliacao.md](../docs/fase5_interface_verificacao_avaliacao.md).

> **Fixtures:** as respostas avaliadas abaixo são fixtures manuais e as notas positivas são o gabarito delas, não métricas do agente.

Notebooks de origem: `incrementos/13_financeiro_confirmado.ipynb`, `incrementos/14_verificacao_avaliacao.ipynb`.

---
*Origem desta parte: `incrementos/13_financeiro_confirmado.ipynb`.*

## Financeiro: previsto, informado e confirmado

Demonstração local com corpus fictício e fixture de extração. Não usa API nem dados da JMS. O projeto operacional é criado a partir de uma interpretação aceita; orçamento e pagamento exigem ações explícitas separadas.


In [1]:
import sys, tempfile
from pathlib import Path
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0,str(raiz/'src'))
from secretaria.banco import Banco, Contexto
from secretaria.extracao import analisar_snapshot
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.fila_revisao import FilaRevisao
from secretaria.projetos_fontes import ProjetosFontes
from secretaria.financeiro import Financeiro


In [2]:
with tempfile.TemporaryDirectory() as pasta:
    with Banco(Path(pasta)/'demo.sqlite3') as banco:
        banco.preparar_demo()
        ana = Contexto('demo','ana')
        corpus = banco.selecionar_corpus(ana,'Demonstração fictícia')
        banco.importar(ana,corpus,raiz/'dados/ficticios/conversa_empresa.txt')
        analise = analisar_snapshot(banco.snapshot_analise(ana,corpus),criar_modelo_fixture(),'fixture_manual')
        fila = FilaRevisao(banco)
        proposta = fila.registrar_resultado(ana,analise)['ids'][0]
        fila.decidir(ana,proposta,1,'aceitar','Fonte conferida para esta demonstração.')
        projetos = ProjetosFontes(banco)
        previa = projetos.prever(ana,proposta)
        projeto_id = projetos.confirmar(ana,proposta,previa['versao'])['projeto_id']
        financeiro = Financeiro(banco)
        orcamento_id = financeiro.criar_orcamento(ana,projeto_id,1_000_000,30)['orcamento_id']
        assert financeiro.relatorio(ana,projeto_id)['projetos'][0]['saldo_centavos'] is None
        financeiro.aceitar_orcamento(ana,orcamento_id)
        entrada, restante = financeiro.definir_parcelas(ana,orcamento_id,[
            {'descricao':'Entrada','valor_centavos':300_000},
            {'descricao':'Restante','valor_centavos':700_000}])['parcelas_ids']
        pagamento_id = financeiro.informar_pagamento(ana,entrada,100_000,'2026-11-11','fixture-1')['pagamento_id']
        antes = financeiro.relatorio(ana,projeto_id)['projetos'][0]
        financeiro.confirmar_pagamento(ana,pagamento_id)
        depois = financeiro.relatorio(ana,projeto_id)['projetos'][0]
        print('Antes da confirmação:',antes['saldo_centavos'],'centavos')
        print('Depois da confirmação:',depois['saldo_centavos'],'centavos')
        assert antes['saldo_centavos'] == 1_000_000
        assert depois['saldo_centavos'] == 900_000


Antes da confirmação: 1000000 centavos
Depois da confirmação: 900000 centavos


> **Como ler o resultado:**
>
> - Orçamento de R$ 10.000,00 (os valores ficam em **centavos**: `1000000` = R$ 10.000,00).
> - `Antes da confirmação: 1000000`: um pagamento de R$ 1.000,00 foi só **informado**; o saldo não muda.
> - `Depois da confirmação: 900000`: depois de **confirmado** por uma pessoa, o saldo cai para R$ 9.000,00.
> - Previsto ≠ informado ≠ confirmado: só o confirmado mexe no saldo.

O percentual acordado de 30% não comprova pagamento. O relato de recebimento no chat também não entra no saldo. A confirmação operacional é registrada em tabela própria com autoria e auditoria.


---
*Origem desta parte: `incrementos/14_verificacao_avaliacao.ipynb`.*

## Verificação e avaliação da Secretária IA
Este notebook usa os mesmos módulos do terminal e do Streamlit. As respostas abaixo são **fixtures manuais**, sem chamada de LLM ou LangSmith.

In [3]:
from pathlib import Path
import sys
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
assert (raiz / 'src' / 'secretaria').exists(), 'Execute na pasta do projeto ou em notebooks/'
sys.path.insert(0, str(raiz / 'src'))
from secretaria.avaliacao import carregar_casos, avaliar_caso
from secretaria.verificador import verificar_deterministico
casos = carregar_casos(raiz)
len(casos)

6

> **Como ler o resultado:**
>
> - `6`: casos de teste fictícios carregados para a avaliação (com gabarito).

In [4]:
resultados = [avaliar_caso(c, c['fixture_resposta'], c['fixture_ferramentas']) for c in casos]
for r in resultados:
    print(r['id'], 'ferramentas=', r['nota_ferramentas'], 'resposta_fixture=', r['nota_resposta'])
assert next(r for r in resultados if r['id'] == 'oferta_consulta')['nota_resposta'] == 0

saudacao ferramentas= 1 resposta_fixture= 1
cliente ferramentas= 1 resposta_fixture= 1
contagem ferramentas= 1 resposta_fixture= 1
fonte ferramentas= 1 resposta_fixture= 1
proposta_visita ferramentas= 1 resposta_fixture= 1
oferta_consulta ferramentas= 0 resposta_fixture= 0


> **Como ler o resultado:**
>
> - Para cada caso: `ferramentas=1` = o agente usou as ferramentas esperadas; `resposta_fixture=1` = a resposta bateu com o gabarito.
> - `oferta_consulta ... 0`: caso feito para **falhar** de propósito, para provar que a avaliação também detecta erro.
> - Aqui as respostas são fixtures. A avaliação real (4 execuções com API e LangSmith) está nas evidências da fase 5.

A igualdade das listas de ferramentas é calculada em Python. A nota de resposta local vem do gabarito da fixture, não de um juiz executado. Na execução com API, um juiz semântico usa critérios explícitos e a revisão humana continua necessária.

In [5]:
import json
from types import SimpleNamespace
retorno = SimpleNamespace(type='tool', name='consultar_mensagens_estruturado', content=json.dumps({'status':'ok','plano':{'tipo':'contagem'},'total':1}))
print(verificar_deterministico('Quantas mensagens?', 'Foram encontradas 3 mensagens.', [retorno], 'consulta'))
print(verificar_deterministico('Quantas mensagens?', 'Foi encontrada 1 mensagem.', [retorno], 'consulta'))

{'aprovada': False, 'problemas': ['Contagem declarada 3 difere do total 1.']}
{'aprovada': True, 'problemas': []}


> **Como ler o resultado:**
>
> - **Verificador determinístico** (regras em Python, antes do juiz semântico):
> - 1ª linha: a resposta diz "3 mensagens", mas a ferramenta contou **1** → `aprovada: False`. Resposta barrada.
> - 2ª linha: a resposta diz "1 mensagem", igual à ferramenta → `aprovada: True`.